# Evaluation der Speaker-Attribution-Pipelines

Dieses Notebook bündelt die finale Auswertung der lokalen Speaker-Attribution-Pipelines auf Basis eines gemeinsamen kanonischen ASR-Transkripts.

Es erzeugt die zentralen Kennzahlen des Papers:
- Word Speaker Error Rate (WSER)
- Validity Rate (VR)
- Speaker Count Accuracy (SCA)
- Real-Time Factor (RTF)

In [ ]:
### Imports + Config
import json
import re
import os
from pathlib import Path
from itertools import permutations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv

## Für Debugs
from collections import defaultdict
from difflib import SequenceMatcher

# Pandas-Options:
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 160)

load_dotenv()

repo_root_env = os.getenv("REPO_ROOT")
assert repo_root_env, "REPO_ROOT ist nicht gesetzt in .env"
REPO_ROOT = Path(repo_root_env).expanduser().resolve()
assert REPO_ROOT.exists(), f"REPO_ROOT existiert nicht: {REPO_ROOT}"

RESULTS_DIR = REPO_ROOT / "results"
assert RESULTS_DIR.exists(), f"RESULTS_DIR existiert nicht: {RESULTS_DIR}"

FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

### Ausgegebenes und annotiertes Label-Muster als Regex
SPEAKER_PATTERN = re.compile(r'\[Sprecher (\d+)\]:\s*')

# Pipeline-Übersicht, macht Auswertung einfacher
PIPELINE_ORDER = ["A1_raw", "A1_seg", "A2_raw", "A2_seg", "B", "C", "D"]
PIPELINE_LABELS = {
    "A1_raw": "A₁: MedGemma (raw)",
    "A1_seg": "A1: MedGemma (seg)",
    "A2_raw": "A₂: Llama-3.1 (raw)",
    "A2_seg": "A2: Llama-3.1 (seg)",
    "B": "B: Pya-Seg",
    "C": "C: Pya-FW",
    "D": "D: Pya-WX",
}
CONDITION_ORDER = ["unknown", "known"]
CONDITION_COLORS = {"unknown": "#D32F2F", "known": "#1565C0"}

In [ ]:
### Parser
# Parst '[Sprecher N]: text' Format.
# Returns: (tokens, labels) — gleich lang, ein Label pro Wort.
def parse_dialogue(text: str) -> tuple[list[str], list[str]]:

    tokens, labels = [], []
    current_speaker = None

    for line in text.strip().split('\n'):
        line = line.strip()
        if not line:
            continue

        m = SPEAKER_PATTERN.match(line)
        if m:
            current_speaker = f"S{m.group(1)}"
            content = SPEAKER_PATTERN.sub('', line).strip()
        else:
            # Zeile ohne Speaker-Label → vorherigen Speaker fortsetzen
            if current_speaker is None:
                current_speaker = "S0"
            content = line
        
        words = content.split()
        tokens.extend(words)
        labels.extend([current_speaker] * len(words))
    
    return tokens, labels

## Speaker-Labels entfernen
def strip_speaker_labels(text: str) -> list[str]:
    return SPEAKER_PATTERN.sub("", text).strip().split()

In [ ]:
### Prüfung, ob LLM Tokenstream verändert hat

def validate_token_stream(ref_text: str, hyp_text: str) -> dict:

    # Speaker-Labels entfernen
    ref_tokens = strip_speaker_labels(ref_text)
    hyp_tokens = strip_speaker_labels(hyp_text)
    
    if ref_tokens == hyp_tokens:
        return {
            "valid": True,
            "n_tokens": len(ref_tokens),
            "n_ref": len(ref_tokens),
            "n_hyp": len(hyp_tokens),
            "insertions": 0,
            "deletions": 0,
            "substitutions": 0,
            "token_error_rate": 0.0,
        }

    sm = SequenceMatcher(None, ref_tokens, hyp_tokens)
    ins, dels, subs = 0, 0, 0

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "insert":
            ins += j2 - j1
        elif tag == "delete":
            dels += i2 - i1
        elif tag == "replace":
            subs += max(i2 - i1, j2 - j1)

    return {
        "valid": False,
        "n_ref": len(ref_tokens),
        "n_hyp": len(hyp_tokens),
        "insertions": ins,
        "deletions": dels,
        "substitutions": subs,
        "token_error_rate": round((ins + dels + subs) / max(len(ref_tokens), 1), 4),
    }


def first_token_mismatch(ref_text: str, hyp_text: str, context: int = 8) -> dict:
    ref_tokens = strip_speaker_labels(ref_text)
    hyp_tokens = strip_speaker_labels(hyp_text)

    sm = SequenceMatcher(None, ref_tokens, hyp_tokens)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        return {
            "tag": tag,
            "ref_span": (i1, i2),
            "hyp_span": (j1, j2),
            "ref_tokens": ref_tokens[i1:i2],
            "hyp_tokens": hyp_tokens[j1:j2],
            "ref_context": ref_tokens[max(0, i1 - context):min(len(ref_tokens), i2 + context)],
            "hyp_context": hyp_tokens[max(0, j1 - context):min(len(hyp_tokens), j2 + context)],
        }

    return {"tag": "equal"}

In [ ]:
### Optimale Speaker-Zuordnung (Permutationen beachten)
def best_speaker_mapping(pred_labels: list[str], ref_labels: list[str]) -> dict:
    """
    Findet optimale 1:1-Zuordnung von pred→ref Speakern via Brute-Force.
    Funktioniert für <=4 Sprecher (4! = 24 Permutationen).
    Bei mehr pred-Speakern als Referenz-Speakern bleiben zusätzliche Speaker unmapped und zählen als Fehler.
    """
    pred_speakers = sorted(set(pred_labels))
    ref_speakers = sorted(set(ref_labels))

    best_mapping = None
    best_correct = -1

    if len(pred_speakers) <= len(ref_speakers):
        for perm in permutations(ref_speakers):
            mapping = {p: r for p, r in zip(pred_speakers, perm[:len(pred_speakers)])}
            correct = sum(
                1 for p, r in zip(pred_labels, ref_labels)
                if mapping.get(p) == r
            )
            if correct > best_correct:
                best_correct = correct
                best_mapping = mapping
    else:
        for perm in permutations(pred_speakers):
            mapping = {p: r for p, r in zip(perm[:len(ref_speakers)], ref_speakers)}
            for p in pred_speakers:
                if p not in mapping:
                    mapping[p] = f"UNMAPPED_{p}"
            correct = sum(
                1 for p, r in zip(pred_labels, ref_labels)
                if mapping.get(p) == r
            )
            if correct > best_correct:
                best_correct = correct
                best_mapping = mapping

    if best_mapping is None:
        best_mapping = {p: p for p in pred_speakers}

    return best_mapping

In [ ]:
### WSER berechnen
def compute_wser(pred_text: str, ref_text: str) -> dict:

    pred_tokens, pred_labels = parse_dialogue(pred_text)
    ref_tokens, ref_labels = parse_dialogue(ref_text)
    
    # Token-Stream muss identisch sein
    if pred_tokens != ref_tokens:
        return {
            "wser": None,
            "error": f"Token mismatch: {len(pred_tokens)} pred vs {len(ref_tokens)} ref",
            "n_total": len(ref_tokens),
            "n_pred_tokens": len(pred_tokens),
            "n_speakers_pred": len(set(pred_labels)),
            "n_speakers_ref": len(set(ref_labels)),
            "mapping": None,
        }
    
    mapping = best_speaker_mapping(pred_labels, ref_labels)
    mapped_pred = [mapping.get(l, l) for l in pred_labels]
    
    n_total = len(ref_labels)
    n_correct = sum(1 for p, r in zip(mapped_pred, ref_labels) if p == r)
    n_errors = n_total - n_correct
    
    return {
        "wser": round(n_errors / n_total, 4) if n_total > 0 else 0.0,
        "n_total": n_total,
        "n_correct": n_correct,
        "n_errors": n_errors,
        "mapping": mapping,
        "n_speakers_pred": len(set(pred_labels)),
        "n_speakers_ref": len(set(ref_labels)),
    }

In [ ]:
### Boundary-WSER berechnen (im Fenster +- k Wörter um Sprecherwechsel), LEGACY, ggf. wichtig für andere Settings in denen WhisperX genauer untersucht wird
def compute_boundary_wser(pred_text: str, ref_text: str, k: int = 2) -> dict:

    pred_tokens, pred_labels = parse_dialogue(pred_text)
    ref_tokens, ref_labels = parse_dialogue(ref_text)

    if pred_tokens != ref_tokens:
        return {
            "boundary_wser": None,
            "n_boundary_words": None,
            "n_boundary_correct": None,
            "n_turns": None,
            "error": "Token mismatch",
        }
    
    mapping = best_speaker_mapping(pred_labels, ref_labels)
    mapped_pred = [mapping.get(l, l) for l in pred_labels]
    
    # Positionen der Sprecherwechsel in der Referenz finden
    boundary_positions = set()
    for i in range(1, len(ref_labels)):
        if ref_labels[i] != ref_labels[i-1]:
            # Fenster +-k um den Wechsel
            for j in range(max(0, i - k), min(len(ref_labels), i + k + 1)):
                boundary_positions.add(j)
    
    if not boundary_positions:
        return {
            "boundary_wser": 0.0,
            "n_boundary_words": 0,
            "n_boundary_correct": 0,
            "n_turns": 0,
            "error": None,
        }    
    n_boundary = len(boundary_positions)
    n_correct = sum(1 for i in boundary_positions if mapped_pred[i] == ref_labels[i])
    
    return {
        "boundary_wser": round((n_boundary - n_correct) / n_boundary, 4),
        "n_boundary_words": n_boundary,
        "n_boundary_correct": n_correct,
        "n_turns": sum(1 for i in range(1, len(ref_labels)) if ref_labels[i] != ref_labels[i-1]),
        "error": None,
    }

In [ ]:
### RTF-Berechnung aus meta.json
def compute_rtf(meta: dict, run_key: str) -> dict:
    audio_dur = meta.get("audio_duration_s")
    if not audio_dur or audio_dur == 0:
        return {"rtf_total": None, "rtf_post_asr": None, "components": {}}

    run = meta.get("runs", {}).get(run_key, {})
    sec_e2e = float(run.get("seconds_e2e", 0.0) or 0.0)
    sec_post = float(run.get("seconds_post_asr", 0.0) or 0.0)

    sec_norm = float(meta.get("seconds", {}).get("normalize", 0.0) or 0.0)
    sec_asr = float(meta.get("seconds", {}).get("asr", 0.0) or 0.0)
    sec_align = float(meta.get("seconds", {}).get("whisperx_align", 0.0) or 0.0)

    is_aligned = ("word_aligned" in run_key.lower()) or ("wx" in run_key.lower())
    sec_diarize = max(0.0, sec_post - (sec_align if is_aligned else 0.0))

    return {
        "rtf_total": round(sec_e2e / audio_dur, 4),
        "rtf_post_asr": round(sec_post / audio_dur, 4),
        "components": {
            "normalization": round(sec_norm / audio_dur, 4),
            "asr": round(sec_asr / audio_dur, 4),
            "alignment": round(sec_align / audio_dur, 4) if is_aligned else 0.0,
            "diarization": round(sec_diarize / audio_dur, 4),
        },
    }

In [ ]:
### Batch-Aggregation
# Standard: echte manuelle Referenz verwenden.
# Nur zum Debuggen von Pipeline-Struktur auf True setzen, wenn noch keine manuelle Referenz vorhanden.
USE_PSEUDO_REF = False


def parse_pipeline(run_key: str) -> str:
    rk = run_key.lower()

    # LLM-Pipelines
    if "medgemma" in rk:
        base = "A1"
        if "segmented" in rk:
            return f"{base}_seg"
        if "raw" in rk:
            return f"{base}_raw"

    if "llama" in rk or "meta-llama" in rk:
        base = "A2"
        if "segmented" in rk:
            return f"{base}_seg"
        if "raw" in rk:
            return f"{base}_raw"
        
    # Pyannote-Pipelines
    if "word_aligned" in rk or "word_alligned" in rk or "pya-wx" in rk:
        return "D"
    if "diarized_word" in rk or "pya-fw" in rk:
        return "C"
    if "diarized_segment" in rk or "pya-seg" in rk:
        return "B"
    return run_key


def parse_condition(run_key: str) -> str:
    rk = run_key.lower()
    if "unknown" in rk:
        return "unknown"
    if "known" in rk:
        return "known"
    return "other"


def add_error(existing: str | None, new_msg: str) -> str:
    if existing in (None, "", "None"):
        return new_msg
    return f"{existing} | {new_msg}"


rows = []
skipped = []

for result_dir in sorted(RESULTS_DIR.iterdir()):
    if not result_dir.is_dir() or result_dir.name == "figures":
        continue

    meta_path = result_dir / "meta.json"
    transcript_path = result_dir / "transcript.txt"
    ref_path = (
        result_dir / "Pyannote_diarized_segment_known.txt"
        if USE_PSEUDO_REF
        else result_dir / "diarization_ref.txt"
    )

    missing = []
    if not meta_path.exists():
        missing.append("meta.json")
    if not transcript_path.exists():
        missing.append("transcript.txt")
    if not ref_path.exists():
        missing.append(ref_path.name)

    if missing:
        skipped.append({"file": result_dir.name, "missing": missing})
        continue

    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    transcript = transcript_path.read_text(encoding="utf-8")
    ref_text = ref_path.read_text(encoding="utf-8")

    try:
        ref_tokens, ref_labels = parse_dialogue(ref_text)
    except Exception as e:
        skipped.append({"file": result_dir.name, "missing": [f"ref parse error: {e!r}"]})
        continue

    spk_match = re.search(r"_(\d)S", result_dir.name)
    n_speakers_from_filename = int(spk_match.group(1)) if spk_match else None

    for run_key, run_info in meta.get("runs", {}).items():
        pipeline = parse_pipeline(run_key)
        condition = parse_condition(run_key)
        output_path = result_dir / f"{run_key}.txt"

        row = {
            "file": result_dir.name,
            "run_key": run_key,
            "pipeline": pipeline,
            "condition": condition,
            "output_exists": output_path.exists(),
            "nonempty": None,
            "valid": False,
            "valid_vs_transcript": None,
            "n_ref_eval": None,
            "n_hyp_eval": None,
            "n_ref_transcript": None,
            "n_hyp_transcript": None,
            "insertions": None,
            "deletions": None,
            "substitutions": None,
            "token_error_rate": None,
            "wser": None,
            "boundary_wser": None,
            "n_words": None,
            "n_boundary_words": None,
            "n_turns": None,
            "n_speakers_from_filename": n_speakers_from_filename,
            "known_num_speakers_meta": meta.get("known_num_speakers"),
            "n_speakers_ref_eval": len(set(ref_labels)),
            "n_speakers_pred": None,
            "sca": None,
            "audio_duration_s": meta.get("audio_duration_s"),
            "rtf_total": None,
            "rtf_post_asr": None,
            "rtf_normalization": None,
            "rtf_asr": None,
            "rtf_alignment": None,
            "rtf_diarization": None,
            "ok": bool(run_info.get("ok", False)),
            "error": run_info.get("error"),
            "ref_source": ref_path.name,
            "mapping": None,
        }

        try:
            rtf = compute_rtf(meta, run_key)
            row["rtf_total"] = rtf["rtf_total"]
            row["rtf_post_asr"] = rtf["rtf_post_asr"]
            for comp_key, comp_val in rtf.get("components", {}).items():
                row[f"rtf_{comp_key}"] = comp_val
        except Exception as e:
            row["error"] = add_error(row["error"], f"RTF: {e!r}")

        if not output_path.exists():
            row["error"] = add_error(row["error"], "missing output file")
            rows.append(row)
            continue

        hyp_text = output_path.read_text(encoding="utf-8")
        row["nonempty"] = bool(hyp_text.strip())
        if not row["nonempty"]:
            row["error"] = add_error(row["error"], "empty output")
            rows.append(row)
            continue

        # Primärer Validity-Check: gegen Evaluationsreferenz.
        validation = validate_token_stream(ref_text, hyp_text)
        row["valid"] = validation["valid"]
        row["n_ref_eval"] = validation.get("n_ref", validation.get("n_tokens"))
        row["n_hyp_eval"] = validation.get("n_hyp", validation.get("n_tokens"))
        row["insertions"] = validation.get("insertions", 0)
        row["deletions"] = validation.get("deletions", 0)
        row["substitutions"] = validation.get("substitutions", 0)
        row["token_error_rate"] = validation.get("token_error_rate", 0.0)

        # Sekundärer Diagnose-Check: gegen kanonisches Transcript.
        validation_transcript = validate_token_stream(transcript, hyp_text)
        row["valid_vs_transcript"] = validation_transcript["valid"]
        row["n_ref_transcript"] = validation_transcript.get("n_ref", validation_transcript.get("n_tokens"))
        row["n_hyp_transcript"] = validation_transcript.get("n_hyp", validation_transcript.get("n_tokens"))

        if row["valid"]:
            wser_result = compute_wser(hyp_text, ref_text)
            if wser_result.get("error") is None:
                row["wser"] = wser_result["wser"]
                row["n_words"] = wser_result["n_total"]
                row["n_speakers_pred"] = wser_result["n_speakers_pred"]
                row["n_speakers_ref_eval"] = wser_result["n_speakers_ref"]
                row["sca"] = float(row["n_speakers_pred"] == row["n_speakers_ref_eval"])
                row["mapping"] = str(wser_result["mapping"])

                bwser_result = compute_boundary_wser(hyp_text, ref_text, k=2)
                if bwser_result.get("error") is None:
                    row["boundary_wser"] = bwser_result["boundary_wser"]
                    row["n_boundary_words"] = bwser_result["n_boundary_words"]
                    row["n_turns"] = bwser_result["n_turns"]
                else:
                    row["error"] = add_error(row["error"], f"Boundary-WSER: {bwser_result['error']}")
            else:
                row["error"] = add_error(row["error"], f"WSER: {wser_result['error']}")

        rows.append(row)

df = pd.DataFrame(rows)

print("=" * 70)
print("Evaluation Summary")
print("=" * 70)
print(f"Referenz-Modus:   {'PSEUDO (Pyannote-Segment)' if USE_PSEUDO_REF else 'MANUELL (diarization_ref.txt)'}")
print(f"Result-Ordner:    {df['file'].nunique() if len(df) else 0}")
print(f"Runs gesamt:      {len(df)}")
print(f"Valide Runs:      {int(df['valid'].fillna(False).sum())} / {len(df)}")
print(f"Runs mit WSER:    {int(df['wser'].notna().sum())}")
print("=" * 70)

if skipped:
    print("\nÜbersprungene Ordner:")
    for item in skipped:
        print(f"  {item['file']}: {item['missing']}")

if len(df):
    print("\nValidity Rate pro Pipeline x Condition:")
    validity = (
        df.groupby(["pipeline", "condition"], dropna=False)["valid"]
        .agg(["sum", "count", "mean"])
        .rename(columns={"sum": "valid_runs", "count": "total_runs", "mean": "valid_rate"})
    )
    print(validity.to_string())

    print("\nRuns mit Fehlern / Hinweisen:")
    err_df = df[df["error"].notna() & (df["error"] != "")].copy()
    if len(err_df) == 0:
        print("Keine Fehler")
    else:
        for _, row in err_df.iterrows():
            print(f"  {row['file']} / {row['run_key']}: {str(row['error'])[:140]}")
else:
    print("Keine Ergebnisse gefunden.")

## Zentrale Paper-Ergebnisse

Die folgenden Zellen erzeugen die wichtigsten quantitativen Ergebnisse des Papers, insbesondere die WSER-Übersichten und die Validity Rate pro Pipeline.
Außerdem wird jeder einzelne Run mit den jeweiligen Informationen ausgegeben.

In [ ]:
### Ergebnis-Tabellen und Diagnose-Views
print(f"Gesamt-Runs: {len(df)}")
print(f"Dateien: {df['file'].nunique() if len(df) else 0}")
print(f"Referenzmodus: {'Pseudo-reference (B)' if USE_PSEUDO_REF else 'Manual reference'}")

print("\n1) Alle Runs")
display(df.sort_values(["file", "pipeline", "condition"]).reset_index(drop=True))

print("\n2) Validity nach Pipeline x Condition")
validity_table = (
    df.groupby(["pipeline", "condition"], dropna=False)["valid"]
    .mean()
    .reset_index(name="valid_rate")
    .sort_values(["pipeline", "condition"])
)
display(validity_table)

print("\n3) Invalid Runs")
invalid_runs = df[df["valid"] == False].copy()
display(
    invalid_runs[
        [
            "file", "run_key", "pipeline", "condition",
            "output_exists", "nonempty", "valid",
            "n_ref_eval", "n_hyp_eval",
            "insertions", "deletions", "substitutions",
            "token_error_rate", "valid_vs_transcript", "error",
        ]
    ].sort_values(["file", "pipeline", "condition"]).reset_index(drop=True)
)

valid_df = df[df["wser"].notna()].copy()

if len(valid_df) == 0:
    print("\n Keine validen WSER-Ergebnisse vorhanden.")
else:
    def median_iqr(x: pd.Series) -> str:
        med = x.median()
        iqr = x.quantile(0.75) - x.quantile(0.25)
        return f"{med:.2f} ({iqr:.2f})"

    print("\n4) Table 2: Median WSER (IQR) nach Pipeline × Sprecherzahl × Condition")
    table_wser = (
        valid_df.groupby(["pipeline", "n_speakers_from_filename", "condition"])["wser"]
        .agg(median_iqr)
        .unstack(["n_speakers_from_filename", "condition"])
    )
    table_wser = table_wser.reindex([p for p in PIPELINE_ORDER if p in table_wser.index])
    display(table_wser)

    print("\n4b) Table 3: LLM-Input-Ablation: raw vs segmented")
    llm_df = valid_df[valid_df["pipeline"].isin(["A1_raw", "A1_seg", "A2_raw", "A2_seg"])].copy()
    if len(llm_df) == 0:
        print("Keine validen LLM-Ergebnisse vorhanden.")
    else:
        llm_table = (
            llm_df.groupby(["pipeline", "n_speakers_from_filename", "condition"])["wser"]
            .agg(median_iqr)
            .unstack(["n_speakers_from_filename", "condition"])
        )
        llm_table = llm_table.reindex([p for p in PIPELINE_ORDER if p in llm_table.index])
        display(llm_table)

        print("\n4c) Delta WSER: segmented - raw")
        llm_df = valid_df[
            valid_df["pipeline"].isin(["A1_raw", "A1_seg", "A2_raw", "A2_seg"])
        ].copy()

        if len(llm_df) == 0:
            print("Keine validen LLM-Ergebnisse vorhanden.")
        else:
            llm_df["llm_model"] = llm_df["pipeline"].str.extract(r"^(A1|A2)_")
            llm_df["input_form"] = llm_df["pipeline"].str.extract(r"_(raw|seg)$")

            delta_base = (
                llm_df.groupby(["llm_model", "input_form", "n_speakers_from_filename", "condition"])["wser"]
                .median()
                .unstack("input_form")
            )

            delta_base = delta_base.dropna(subset=["raw", "seg"]).copy()
            delta_base["delta_seg_minus_raw"] = delta_base["seg"] - delta_base["raw"]

            display(delta_base[["raw", "seg", "delta_seg_minus_raw"]].round(3))



    print("\n5) Table 4: WhisperX-Ablation (C vs D)")
    cd_df = valid_df[valid_df["pipeline"].isin(["C", "D"]) & valid_df["boundary_wser"].notna()].copy()
    if len(cd_df) == 0:
        print("Keine validen C/D-Ergebnisse vorhanden.")
    else:
        table_wx = (
            cd_df.groupby(["pipeline", "condition"])[["wser", "boundary_wser"]]
            .median()
            .round(3)
        )
        display(table_wx)

        for cond in CONDITION_ORDER:
            bwser_c = cd_df[(cd_df["pipeline"] == "C") & (cd_df["condition"] == cond)]["boundary_wser"].median()
            bwser_d = cd_df[(cd_df["pipeline"] == "D") & (cd_df["condition"] == cond)]["boundary_wser"].median()
            if pd.notna(bwser_c) and bwser_c > 0 and pd.notna(bwser_d):
                delta = (bwser_d - bwser_c) / bwser_c * 100
                print(f"Δ Boundary-WSER {cond}: {delta:+.1f}% relativ")

    print("\n6) Table 5: Median RTF")
    rtf_df = df[df["rtf_total"].notna()].copy()
    table_rtf = (
        rtf_df.groupby(["pipeline", "condition"])[["rtf_total", "rtf_post_asr"]]
        .median()
        .round(3)
    )
    desired_idx = [(p, c) for p in PIPELINE_ORDER for c in CONDITION_ORDER if (p, c) in table_rtf.index]
    table_rtf = table_rtf.reindex(desired_idx)
    display(table_rtf)

    print("\n7) LLM Output Validity Rate")
    llm_df = df[df["pipeline"].isin(["A1_raw", "A1_seg", "A2_raw", "A2_seg"])].copy()
    if len(llm_df):
        llm_validity = llm_df.groupby("pipeline")["valid"].agg(["sum", "count"])
        llm_validity["rate"] = (llm_validity["sum"] / llm_validity["count"]).round(3)
        display(llm_validity)

    print("\n4d) Delta WSER: known - unknown")
    cond_base = (
        valid_df.groupby(["pipeline", "n_speakers_from_filename", "condition"])["wser"]
        .median()
        .unstack("condition")
    )

    cond_base = cond_base.dropna(subset=["unknown", "known"]).copy()
    cond_base["delta_known_minus_unknown"] = cond_base["known"] - cond_base["unknown"]

    display(cond_base[["unknown", "known", "delta_known_minus_unknown"]].round(3))

In [ ]:
### Zusammenfassung
print("=" * 70)
print("ZUSAMMENFASSUNG")
print("=" * 70)

valid_df = df[df["wser"].notna()].copy()

if len(valid_df) == 0:
    print("Keine validen Ergebnisse.")
else:
    print(f"\nDateien: {valid_df['file'].nunique()} | Runs mit WSER: {len(valid_df)}")
    print(f"Referenz: {'Pseudo (Pyannote-Segment)' if USE_PSEUDO_REF else 'Manuell (diarization_ref.txt)'}")

    print("\nBeste Pipeline pro Sprecherzahl (known, Median WSER)")
    for n_spk in sorted(valid_df["n_speakers_from_filename"].dropna().unique()):
        sub = valid_df[
            (valid_df["n_speakers_from_filename"] == n_spk) &
            (valid_df["condition"] == "known")
        ]
        if len(sub) == 0:
            continue
        best = sub.groupby("pipeline")["wser"].median().sort_values()
        best_name = best.index[0]
        best_val = best.iloc[0]
        print(f"  {int(n_spk)} Sprecher: {PIPELINE_LABELS.get(best_name, best_name)} -> WSER {best_val:.3f}")

    print("\nLLM vs. bestes Pyannote (known, Median WSER)")
    for n_spk in sorted(valid_df["n_speakers_from_filename"].dropna().unique()):
        sub = valid_df[
            (valid_df["n_speakers_from_filename"] == n_spk) &
            (valid_df["condition"] == "known")
        ]
        if len(sub) == 0:
            continue
        
        llm_best = sub[
            sub["pipeline"].isin(["A1_raw", "A1_seg", "A2_raw", "A2_seg"])
        ].groupby("pipeline")["wser"].median().min()

        pya_best = sub[
            sub["pipeline"].isin(["B", "C", "D"])
        ].groupby("pipeline")["wser"].median().min()
        
        if pd.notna(llm_best) and pd.notna(pya_best):
            delta = llm_best - pya_best
            print(f"  {int(n_spk)} Sprecher: LLM {llm_best:.3f} vs Pyannote {pya_best:.3f} (Δ = {delta:+.3f})")

    print("\nValidity pro Pipeline (gegen Referenz)")
    validity = (
        df.groupby(["pipeline", "condition"])["valid"]
        .mean()
        .unstack("condition")
        .reindex(PIPELINE_ORDER)
        .round(3)
    )
    display(validity)

    print("\nRTF-Übersicht (unknown, Median)")
    rtf_summary = df[df["rtf_total"].notna() & (df["condition"] == "unknown")]
    for p in PIPELINE_ORDER:
        vals = rtf_summary[rtf_summary["pipeline"] == p]["rtf_total"]
        if len(vals) > 0:
            print(f"  {PIPELINE_LABELS.get(p, p)}: RTF {vals.median():.2f}")

## Ergänzende Auswertungen

Dieser Abschnitt enthält ergänzende Analysen für das Paper, insbesondere:
- Speaker Count Accuracy (SCA)
- die physician–patient subgroup (`ärztesprech_...`)

In [ ]:
### SCA

sca_df = df[df["sca"].notna()].copy()

print("\n" + "=" * 70)
print("Speaker Count Accuracy (SCA)")
print("=" * 70)

sca_pipeline = (
    sca_df
    .groupby(["pipeline", "n_speakers_from_filename", "condition"])["sca"]
    .agg(n_correct="sum", n_eval="count", sca="mean")
    .reset_index()
    .sort_values(["pipeline", "n_speakers_from_filename", "condition"])
)
display(sca_pipeline.round(3))

# Modellfamilie ableiten
def model_family(p):
    if p in ["A1_raw", "A1_seg"]:
        return "MedGemma"
    if p in ["A2_raw", "A2_seg"]:
        return "Llama"
    return p

sca_df["model_family"] = sca_df["pipeline"].map(model_family)

print("=" * 80)
print("SCA model-aggregated (weighted by file count)")
print("=" * 80)

sca_model_weighted = (
    sca_df[sca_df["model_family"].isin(["MedGemma", "Llama"])]
    .groupby(["model_family", "n_speakers_from_filename", "condition"])["sca"]
    .agg(n_correct="sum", n_eval="count", sca="mean")
    .reset_index()
    .sort_values(["model_family", "n_speakers_from_filename", "condition"])
)
display(sca_model_weighted.round(3))

print("=" * 80)
print("SCA model-aggregated (unweighted mean of raw + seg)")
print("=" * 80)

sca_model_unweighted = (
    sca_pipeline[sca_pipeline["pipeline"].isin(["A1_raw", "A1_seg", "A2_raw", "A2_seg"])]
    .assign(model_family=lambda x: x["pipeline"].map(model_family))
    .groupby(["model_family", "n_speakers_from_filename", "condition"])["sca"]
    .mean()
    .reset_index()
    .sort_values(["model_family", "n_speakers_from_filename", "condition"])
)
display(sca_model_unweighted.round(3))

print("=" * 80)
print("Paper-style compact SCA strings (3S / 4S)")
print("=" * 80)

for model_name, sub in sca_model_unweighted.groupby("model_family"):
    vals = {}
    for spk in [3, 4]:
        for cond in ["known", "unknown"]:
            m = sub[
                (sub["n_speakers_from_filename"] == spk) &
                (sub["condition"] == cond)
            ]["sca"]
            vals[(spk, cond)] = m.iloc[0] if len(m) else np.nan

    print(
        f"{model_name}: "
        f"known 3S/4S = {vals[(3, 'known')]:.3f}/{vals[(4, 'known')]:.3f}; "
        f"unknown 3S/4S = {vals[(3, 'unknown')]:.3f}/{vals[(4, 'unknown')]:.3f}"
    )

In [ ]:
### Physician-Patient (Ärztesprech) subset

import pandas as pd

arzt_df = df[
    df["file"].str.lower().str.startswith("ärztesprech_")
    & df["valid"].fillna(False)
    & df["wser"].notna()
].copy()

print("=" * 80)
print("Physician–patient subgroup (files starting with 'ärztesprech_')")
print("=" * 80)
print(f"n files: {arzt_df['file'].nunique()}")
print()

subgroup_summary = (
    arzt_df
    .groupby(["pipeline", "condition"])["wser"]
    .median()
    .unstack("condition")
)

subgroup_summary = subgroup_summary.reindex(PIPELINE_ORDER)

display(subgroup_summary.round(3))

## Finale Abbildungen

Die folgenden Zellen erzeugen die finalen Abbildungen des Papers.

In [ ]:
### Figure 1 final, Pipeline-Design
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

FIGURES_DIR = Path(FIGURES_DIR) if "FIGURES_DIR" in globals() else Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "figure.dpi": 250,
})

fig, ax = plt.subplots(figsize=(14.2, 6.0))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

# Colors
C_NEUTRAL   = "#555555"
C_FW        = "#0D4991"   # darker blue
C_FW_LIGHT  = "#6D9DC5"
C_PYA       = "#1B5E20"   # dark green
C_PYA_LIGHT = "#A5D6A7"
C_WX        = "#777777"

ARROW_MS = 15
INNER = 0.008

# Helpers
def box(x, y, w, h, text, fc="#f7f7f7", ec=C_NEUTRAL, lw=1.0, ls="-", fontsize=8.6, z=3):
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.01,rounding_size=0.015",
        linewidth=lw, linestyle=ls,
        edgecolor=ec, facecolor=fc,
        zorder=z,
    )
    ax.add_patch(patch)
    ax.text(
        x + w / 2, y + h / 2, text,
        ha="center", va="center",
        fontsize=fontsize, zorder=z + 1
    )
    return {"x": x, "y": y, "w": w, "h": h, "patch": patch}

def pt(b, side, frac=0.5):
    if side == "left":
        return (b["x"], b["y"] + b["h"] * frac)
    if side == "right":
        return (b["x"] + b["w"], b["y"] + b["h"] * frac)
    if side == "top":
        return (b["x"] + b["w"] * frac, b["y"] + b["h"])
    if side == "bottom":
        return (b["x"] + b["w"] * frac, b["y"])
    raise ValueError(side)

def inner_pt(b, side, frac=0.5, delta=INNER):
    if side == "left":
        return (b["x"] + delta, b["y"] + b["h"] * frac)
    if side == "right":
        return (b["x"] + b["w"] - delta, b["y"] + b["h"] * frac)
    if side == "top":
        return (b["x"] + b["w"] * frac, b["y"] + b["h"] - delta)
    if side == "bottom":
        return (b["x"] + b["w"] * frac, b["y"] + delta)
    raise ValueError(side)

def line_segment(p0, p1, color=C_NEUTRAL, lw=1.2, ls="-", z=2, patchA=None, patchB=None):
    ax.add_patch(FancyArrowPatch(
        p0, p1,
        arrowstyle='-',
        mutation_scale=ARROW_MS,
        linewidth=lw,
        linestyle=ls,
        color=color,
        shrinkA=0, shrinkB=0,
        patchA=patchA, patchB=patchB,
        zorder=z
    ))

def arrow_segment(p0, p1, color=C_NEUTRAL, lw=1.2, ls="-", z=7, patchA=None, patchB=None):
    ax.add_patch(FancyArrowPatch(
        p0, p1,
        arrowstyle='-|>',
        mutation_scale=ARROW_MS,
        linewidth=lw,
        linestyle=ls,
        color=color,
        shrinkA=0, shrinkB=0,
        patchA=patchA, patchB=patchB,
        zorder=z
    ))

def route_line(points, color="#B0B0B0", lw=1.2, ls="-", z=1):
    for (x0, y0), (x1, y1) in zip(points[:-1], points[1:]):
        ax.plot(
            [x0, x1], [y0, y1],
            color=color, linewidth=lw, linestyle=ls,
            solid_capstyle="round", zorder=z
        )

def orth_arrow(points, color=C_NEUTRAL, lw=1.2, ls="-", z=4,
               label=None, label_xy=None, label_color=None, label_fs=7.5,
               patchA=None, patchB=None):
    if len(points) < 2:
        return

    if len(points) == 2:
        arrow_segment(points[0], points[1], color=color, lw=lw, ls=ls, z=z+2, patchA=patchA, patchB=patchB)
    else:
        for i, ((x0, y0), (x1, y1)) in enumerate(zip(points[:-2], points[1:-1])):
            line_segment(
                (x0, y0), (x1, y1),
                color=color, lw=lw, ls=ls, z=z,
                patchA=patchA if i == 0 else None,
                patchB=None
            )
        arrow_segment(points[-2], points[-1], color=color, lw=lw, ls=ls, z=z+2, patchA=None, patchB=patchB)

    if label is not None and label_xy is not None:
        ax.text(
            label_xy[0], label_xy[1], label,
            ha="center", va="center",
            fontsize=label_fs,
            color=(label_color or color),
            zorder=z + 3
        )

def left_entry_from_bus(bus_x, box_obj, frac=0.5, pre=0.04,
                        color=C_NEUTRAL, lw=1.2, ls="-",
                        label=None, label_xy=None, label_color=None, label_fs=7.5):
    y = pt(box_obj, "left", frac)[1]
    x_end = pt(box_obj, "left", frac)[0]
    x_pre = x_end - pre
    route_line([(bus_x, y), (x_pre, y)], color=color, lw=lw, ls=ls, z=2)
    arrow_segment((x_pre, y), inner_pt(box_obj, "left", frac), color=color, lw=lw, ls=ls, z=7, patchB=box_obj["patch"])
    if label is not None and label_xy is not None:
        ax.text(
            label_xy[0], label_xy[1], label,
            ha="center", va="center",
            fontsize=label_fs,
            color=(label_color or color),
            zorder=8
        )

def rightward_to_output(box_obj, out_box, via_x, color=C_NEUTRAL, lw=1.2, ls="-",
                        frac=0.5, out_frac=0.5, pre=0.03):
    src_inside = inner_pt(box_obj, "right", frac)
    src_edge_y = pt(box_obj, "right", frac)[1]
    out_in = pt(out_box, "left", out_frac)
    pre_pt = (out_in[0] - pre, out_in[1])

    line_segment(src_inside, (via_x, src_edge_y), color=color, lw=lw, ls=ls, z=2, patchA=box_obj["patch"])
    route_line([(via_x, src_edge_y), (via_x, out_in[1]), pre_pt], color=color, lw=lw, ls=ls, z=2)
    arrow_segment(pre_pt, inner_pt(out_box, "left", out_frac), color=color, lw=lw, ls=ls, z=7, patchB=out_box["patch"])

## Boxes

# Audio conversion deliberately farther left
conv = box(0.03, 0.49, 0.14, 0.11, "Audio conversion\nWAV 16 kHz", fc="#F6F6F6", fontsize=11.8)

# Input file centered exactly above Audio conversion
inp_center_x = conv["x"] + conv["w"] / 2
inp_w = 0.12
inp = box(inp_center_x - inp_w / 2, 0.81, inp_w, 0.10, "Input file", fc="#F6F6F6", fontsize=12.0)

fw   = box(0.24, 0.71, 0.15, 0.11, "ASR\nfaster-whisper\nlarge-v3", fc="#D8EAF9", ec=C_NEUTRAL, fontsize=11.8)
pya  = box(0.24, 0.27, 0.15, 0.11, "Pyannote\ncommunity-1", fc="#D9EDD8", ec=C_NEUTRAL, fontsize=11.8)

a1   = box(0.70, 0.76, 0.13, 0.09, "A1\nMedGemma-27B-IT", fc="#F9CD55", fontsize=11.8)
a2   = box(0.70, 0.62, 0.13, 0.09, "A2\nLlama-3.1-8B", fc="#F9CD55", fontsize=11.8)

b    = box(0.70, 0.41, 0.13, 0.09, "B\nSegment overlap", fc="#D9EDD8", ec=C_NEUTRAL, fontsize=11.1)
c    = box(0.70, 0.26, 0.13, 0.09, "C\nWord Viterbi", fc="#D9EDD8", ec=C_NEUTRAL, fontsize=10.9)
d    = box(0.70, 0.11, 0.13, 0.09, "D\nWord Viterbi\naligned", fc="#EEEEEE", ec=C_WX, ls="--", fontsize=10.8)

out  = box(0.91, 0.43, 0.07, 0.14, "Attributed\ntranscript", fc="#F6F6F6", fontsize=11.2)
met  = box(0.91, 0.19, 0.07, 0.14, "WSER\nValidity Rate\nRTF\nSCA", fc="#F6F6F6", fontsize=10.7)

# Input -> conversion (straight vertical, centered)
orth_arrow(
    [inner_pt(inp, "bottom", 0.5), inner_pt(conv, "top", 0.5)],
    color=C_NEUTRAL,
    patchA=inp["patch"],
    patchB=conv["patch"]
)

# Conversion -> faster-whisper / pyannote (orthogonal)

elbow_x = 0.20 

orth_arrow(
    [
        inner_pt(conv, "right", 0.72),
        (elbow_x, pt(conv, "right", 0.72)[1]),
        (elbow_x, pt(fw, "left")[1]),
        inner_pt(fw, "left")
    ],
    color=C_NEUTRAL,
    patchA=conv["patch"],
    patchB=fw["patch"]
)

orth_arrow(
    [
        inner_pt(conv, "right", 0.28),
        (elbow_x, pt(conv, "right", 0.28)[1]),
        (elbow_x, pt(pya, "left")[1]),
        inner_pt(pya, "left")
    ],
    color=C_NEUTRAL,
    patchA=conv["patch"],
    patchB=pya["patch"]
)

# WhisperX dependency from audio conversion -> D
orth_arrow(
    [
        inner_pt(conv, "bottom", 0.75),
        (pt(conv, "bottom", 0.75)[0], 0.06),
        (0.64, 0.06),
        (0.64, pt(d, "left")[1]),
        (pt(d, "left")[0] - 0.04, pt(d, "left")[1]),
        inner_pt(d, "left"),
    ],
    color=C_WX, ls="--",
    label="WhisperX alignment",
    label_xy=(0.39, 0.035),
    label_color=C_WX,
    label_fs=10.8,
    patchA=conv["patch"],
    patchB=d["patch"]
)

# Shared faster-whisper routing bus
fw_out = pt(fw, "right")
fw_bus_x = 0.54

fw_target_ys = [
    fw_out[1],
    pt(a1, "left")[1],
    pt(a2, "left")[1],
    pt(b, "left")[1],
    pt(c, "left")[1],
    pt(d, "left")[1],
]

orth_arrow(
    [inner_pt(fw, "right"), (fw_bus_x, fw_out[1])],
    color=C_FW,
    label="canonical transcript",
    label_xy=((fw["x"] + fw["w"] + fw_bus_x) / 2 - 0.002, fw_out[1] + 0.026),
    label_color=C_FW,
    label_fs=11.0,
    patchA=fw["patch"]
)

route_line([(fw_bus_x, min(fw_target_ys)), (fw_bus_x, max(fw_target_ys))], color=C_FW, lw=1.35)

left_entry_from_bus(
    fw_bus_x, a1, frac=0.50, pre=0.04, color=C_FW, lw=1.25,
    label="raw / segmented input",
    label_xy=(0.62, pt(a1, "left")[1] + 0.018),
    label_color="#B61538", label_fs=10.4
)

left_entry_from_bus(
    fw_bus_x, a2, frac=0.50, pre=0.04, color=C_FW, lw=1.25,
    label="raw / segmented input",
    label_xy=(0.62, pt(a2, "left")[1] + 0.018),
    label_color="#B61538", label_fs=10.4
)

left_entry_from_bus(
    fw_bus_x, b, frac=0.50, pre=0.04, color=C_FW, lw=1.25,
    label="segments",
    label_xy=(0.605, pt(b, "left")[1] + 0.017),
    label_color=C_FW, label_fs=10.5
)

left_entry_from_bus(
    fw_bus_x, c, frac=0.50, pre=0.04, color=C_FW, lw=1.25,
    label="word list + timestamps",
    label_xy=(0.62, pt(c, "left")[1] + 0.018),
    label_color=C_FW, label_fs=10.1
)

left_entry_from_bus(
    fw_bus_x, d, frac=0.50, pre=0.04, color=C_FW, lw=1.25,
    label="word list",
    label_xy=(0.605, pt(d, "left")[1] + 0.017),
    label_color=C_FW, label_fs=10.5
)

# Shared Pyannote routing bus
pya_out = pt(pya, "right")
pya_bus_x = 0.55

pya_target_ys = [
    pya_out[1],
    pt(b, "left", 0.28)[1],
    pt(c, "left", 0.28)[1],
    pt(d, "left", 0.28)[1],
]

orth_arrow(
    [inner_pt(pya, "right"), (pya_bus_x, pya_out[1])],
    color=C_PYA,
    label="exclusive speaker turns",
    label_xy=((pya["x"] + pya["w"] + pya_bus_x) / 2 - 0.002, pya_out[1] + 0.024),
    label_color=C_PYA,
    label_fs=10.8,
    patchA=pya["patch"]
)

route_line([(pya_bus_x, min(pya_target_ys)), (pya_bus_x, max(pya_target_ys))], color=C_PYA, lw=1.35)

left_entry_from_bus(pya_bus_x, b, frac=0.28, pre=0.024, color=C_PYA, lw=1.25)
left_entry_from_bus(pya_bus_x, c, frac=0.28, pre=0.024, color=C_PYA, lw=1.25)
left_entry_from_bus(pya_bus_x, d, frac=0.28, pre=0.024, color=C_PYA, lw=1.25)

# Pipelines -> attributed transcript
collector_x = 0.875

for src in [a1, a2, b, c]:
    rightward_to_output(src, out, via_x=collector_x, color=C_NEUTRAL, lw=1.2, ls="-", frac=0.5, out_frac=0.5)

rightward_to_output(d, out, via_x=collector_x, color=C_WX, lw=1.2, ls="--", frac=0.5, out_frac=0.5)

# Attributed transcript -> metrics
orth_arrow(
    [
        inner_pt(out, "bottom"),
        (pt(out, "bottom")[0], 0.36),
        (pt(met, "top")[0], 0.36),
        inner_pt(met, "top"),
    ],
    color=C_NEUTRAL,
    patchA=out["patch"],
    patchB=met["patch"]
)

# Save
plt.tight_layout()

fig1_path = FIGURES_DIR / "fig1_pipeline_design.png"
plt.savefig(fig1_path, dpi=250, bbox_inches="tight")
print(f"Saved: {fig1_path}")
plt.show()

In [ ]:
### Figure 2: WSER bar plot and Figure 3: RTF

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory

FIGURES_DIR = Path(FIGURES_DIR) if "FIGURES_DIR" in globals() else Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 250,
})

# Common config
PLOT_PIPELINES = ["A1_raw", "A1_seg", "A2_raw", "A2_seg", "B", "C"]
PLOT_LABELS = {
    "A1_raw": "A1\nraw",
    "A1_seg": "A1\nseg",
    "A2_raw": "A2\nraw",
    "A2_seg": "A2\nseg",
    "B": "B",
    "C": "C",
}
SPK_GROUPS = [2, 3, 4]
CONDS = ["unknown", "known"]
WSER_COLORS = {
    "unknown": "#c85a5a",
    "known":   "#4b79c9", 
}

# FIGURE 2 — Median WSER by pipeline and speaker count
valid_df = df[df["wser"].notna()].copy()

fig, axes = plt.subplots(1, 3, figsize=(8.9, 3.1), sharey=True)

for ax, n_spk in zip(axes, SPK_GROUPS):
    sub = valid_df[
        (valid_df["n_speakers_from_filename"] == n_spk)
        & (valid_df["pipeline"].isin(PLOT_PIPELINES))
    ].copy()

    medians = (
        sub.groupby(["pipeline", "condition"])["wser"]
        .median()
        .unstack("condition")
        .reindex(PLOT_PIPELINES)
    )

    x = np.arange(len(PLOT_PIPELINES))
    width = 0.36

    unk_vals = medians["unknown"].fillna(0).to_numpy() if "unknown" in medians.columns else np.zeros(len(PLOT_PIPELINES))
    kn_vals  = medians["known"].fillna(0).to_numpy()   if "known" in medians.columns else np.zeros(len(PLOT_PIPELINES))

    ax.bar(
        x - width / 2, unk_vals, width,
        label="unknown",
        color=WSER_COLORS["unknown"],
    )
    ax.bar(
        x + width / 2, kn_vals, width,
        label="known",
        color=WSER_COLORS["known"],
    )

    ax.set_title(f"{n_spk} speakers", pad=6, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([PLOT_LABELS[p] for p in PLOT_PIPELINES])
    ax.set_ylim(0, 0.68)
    ax.grid(axis="y", alpha=0.25)

    # Separator and group labels
    ax.axvline(3.5, linestyle=":", linewidth=1, color="#777777")
    ax.text(1.5, 0.655, "LLM", ha="center", va="top", fontsize=8, color="#555555")
    ax.text(4.5, 0.655, "Pyannote", ha="center", va="top", fontsize=8, color="#555555")

axes[0].set_ylabel("Median WSER")
axes[-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1.00), frameon=False, fontsize=8)

plt.tight_layout()

fig2_path = FIGURES_DIR / "fig2_wser_barplot.png"
plt.savefig(fig2_path, dpi=250, bbox_inches="tight")
print(f"Saved: {fig2_path}")
plt.show()

# FIGURE 3 — Median end-to-end RTF
from matplotlib.transforms import blended_transform_factory

rtf_df = df[df["rtf_total"].notna()].copy()

def collapse_pipeline_name(p):
    if isinstance(p, str):
        if p.startswith("A1_"):
            return "A1"
        if p.startswith("A2_"):
            return "A2"
        if p in {"B", "C", "D"}:
            return p
    return p

rtf_df["pipeline_base"] = rtf_df["pipeline"].map(collapse_pipeline_name)

RTF_ORDER = ["A1", "A2", "B", "C", "D"]
RTF_COLORS = {
    "A1": "#F9CD55",
    "A2": "#F9CD55",
    "B":  "#75b79f",
    "C":  "#75b79f",
    "D":  "#b8b8b8",
}

rtf_medians = (
    rtf_df.groupby("pipeline_base")["rtf_total"]
    .median()
    .reindex(RTF_ORDER)
)

# keep only pipelines that are actually present
rtf_medians = rtf_medians[rtf_medians.notna()]
rtf_order_present = list(rtf_medians.index)

fig, ax = plt.subplots(figsize=(6.6, 2.8))

x = np.arange(len(rtf_order_present))
vals = rtf_medians.to_numpy()

bars = ax.bar(
    x,
    vals,
    color=[RTF_COLORS[p] for p in rtf_order_present],
)

# Real-time threshold line
ax.axhline(1.0, linestyle="--", color="#555555", linewidth=1)

# Shared ASR reference line
asr_ref = None
if "rtf_asr" in rtf_df.columns and rtf_df["rtf_asr"].notna().any():
    asr_ref = (
        rtf_df.groupby("pipeline_base")["rtf_asr"]
        .median()
        .reindex(rtf_order_present)
        .median()
    )
    if pd.notna(asr_ref):
        ax.axhline(asr_ref, linestyle="--", color="#C62828", linewidth=1)

ax.set_ylabel("Median end-to-end RTF")
ax.set_xticks(x)
ax.set_xticklabels(rtf_order_present)

upper = max(1.08, float(vals.max()) + 0.08)
ax.set_ylim(0, upper)
ax.grid(axis="y", alpha=0.25)

for rect, val in zip(bars, vals):
    ax.text(
        rect.get_x() + rect.get_width() / 2,
        val + 0.012,
        f"{val:.2f}",
        ha="center",
        va="bottom",
        fontsize=8,
        fontweight="bold",
    )

# Labels outside the plot, aligned to line height
trans = blended_transform_factory(ax.transAxes, ax.transData)

ax.text(
    1.02, 1.0,
    "real-time threshold",
    transform=trans,
    ha="left",
    va="center",
    fontsize=8,
    color="#555555",
    clip_on=False,
)

if asr_ref is not None and pd.notna(asr_ref):
    ax.text(
        1.02, asr_ref,
        f"shared ASR ≈ {asr_ref:.02f}",
        transform=trans,
        ha="left",
        va="center",
        fontsize=8,
        color="#C62828",
        clip_on=False,
    )

# Leave space on the right for outside labels
fig.subplots_adjust(right=0.78)

fig3_path = FIGURES_DIR / "fig3_rtf_barplot.png"
plt.savefig(fig3_path, dpi=250, bbox_inches="tight")
print(f"Saved: {fig3_path}")
plt.show()

In [ ]:
# Check, dass Referenzen korrekt sind
for result_dir in sorted(RESULTS_DIR.iterdir()):
    if not result_dir.is_dir():
        continue
    transcript_path = result_dir / "transcript.txt"
    ref_path = result_dir / "diarization_ref.txt"
    if transcript_path.exists() and ref_path.exists():
        transcript = transcript_path.read_text(encoding="utf-8")
        ref_text = ref_path.read_text(encoding="utf-8")
        v = validate_token_stream(transcript, ref_text)
        if not v["valid"]:
            print("Mismatch:", result_dir.name, v)